# F9 homework: an MLP from scratch, then in PyTorch

In **Part A** you build and train a multilayer perceptron in pure NumPy on scikit-learn's handwritten
digits (8×8 images), and prove your backpropagation is right with a gradient check. In **Part B** you
rebuild the same network in PyTorch and then train a bigger one on **MNIST** (28×28 digits) to at least
97% test accuracy.

**Where to run it.** Google Colab is the easiest: PyTorch and torchvision are preinstalled. A GPU is
optional; the whole notebook runs on Colab's free CPU runtime in about 10 minutes, or about 3 minutes on a
T4 GPU (*Runtime → Change runtime type → T4 GPU*). Part A also runs in local Jupyter with NumPy,
matplotlib and scikit-learn.

Every task has `TODO` cells, followed by a check cell that prints `OK` when your code is right. Run the
cells in order, and restart and run all at the end.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

## Part A: a NumPy MLP

### The data (given)

1,797 images of handwritten digits, each 8×8 = 64 pixel values from 0 to 16. We scale the pixels to 0..1
and hold out 20% as a test set. `stratify=y` keeps the same share of each digit in both parts.

In [ ]:
digits = load_digits()
X = digits.data / 16.0
y = digits.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
print(X_train.shape, X_test.shape, np.bincount(y_train))

### Task A1: forward pass and backpropagation

Write three functions for an MLP with ReLU hidden layers and a softmax cross-entropy output, for **any**
list of layer sizes such as `[64, 64, 10]`. Use the course convention: `params` is a list of `[W, b]`
pairs with `W` of shape `(d_in, d_out)`.

1. `init_params(sizes, rng)`: He initialisation, `W` from a normal distribution with standard deviation
   $\sqrt{2/d_\text{in}}$, and zero biases.
2. `forward(params, X)`: returns `(logits, inputs)`, where `inputs[i]` is the input to layer `i` (you
   need it for $dW = A^\top dZ$). ReLU after every layer except the last.
3. `loss_and_grads(params, X, y)`: returns `(loss, grads)`: the mean cross-entropy, computed stably with
   log-sum-exp, and a list of `[dW, db]` pairs in the same order as `params`.

In [ ]:
def init_params(sizes, rng):
    # TODO
    pass

def forward(params, X):
    # TODO
    pass

def loss_and_grads(params, X, y):
    # TODO
    pass

In [ ]:
# Check A1
_p = init_params([64, 32, 10], np.random.default_rng(0))
assert len(_p) == 2 and _p[0][0].shape == (64, 32) and _p[0][1].shape == (32,) and _p[1][0].shape == (32, 10)
assert abs(_p[0][0].std() - np.sqrt(2 / 64)) < 0.02, "use He initialisation: std sqrt(2 / d_in)"
_logits, _inputs = forward(_p, X_train[:5])
assert _logits.shape == (5, 10) and len(_inputs) == 2 and _inputs[1].shape == (5, 32)
_loss, _g = loss_and_grads(_p, X_train[:50], y_train[:50])
assert all(gw.shape == w.shape and gb.shape == b.shape for (w, b), (gw, gb) in zip(_p, _g)), "each gradient must match its parameter's shape"
assert abs(_loss - np.log(10)) < 0.5, f"an untrained 10-class model should start near ln 10 = 2.30, got {_loss:.3f}"
print(f"initial loss {_loss:.3f} (ln 10 = {np.log(10):.3f})")
print("OK")

### Task A2: gradient check

Write `grad_check(f, params, X, y, eps=1e-6, n_checks=30, seed=0)`, where `f` is a function like
`loss_and_grads`. It compares `n_checks` randomly chosen gradient entries with central differences and
returns the **largest** relative error
$\frac{|g_\text{analytic} - g_\text{numeric}|}{|g_\text{analytic}| + |g_\text{numeric}|}$
(treat 0/0 as 0).

For each check: pick a random layer `i`, `j` in `{0, 1}` (weight or bias) and a random index into that
array; nudge the entry by $\pm\varepsilon$, re-evaluate the loss with `f`, and **restore** the entry.

The check cell also feeds your checker a deliberately broken backward pass, which it must catch.

In [ ]:
def grad_check(f, params, X, y, eps=1e-6, n_checks=30, seed=0):
    # TODO
    pass

In [ ]:
# Check A2
_p = init_params([64, 16, 16, 10], np.random.default_rng(1))
_before = [w.copy() for w, _ in _p]
_err = grad_check(loss_and_grads, _p, X_train[:20], y_train[:20])
assert all(np.array_equal(a, w) for a, (w, _) in zip(_before, _p)), "grad_check must restore every parameter it nudges"
print(f"worst relative error: {_err:.2e}")
assert _err < 1e-4, "your backward pass disagrees with finite differences"

def _broken(params, X, y):          # forgets to divide one gradient by n
    loss, grads = loss_and_grads(params, X, y)
    grads[0][0] = grads[0][0] * len(y)
    return loss, grads
assert grad_check(_broken, _p, X_train[:20], y_train[:20], n_checks=60) > 1e-2, "grad_check should catch a broken gradient"
print("OK")

### Task A3: train it

Write `train_numpy(sizes, epochs=30, lr=0.1, batch=32, seed=0)` that initialises a network with
`np.random.default_rng(seed)` and trains it on `X_train, y_train` with mini-batch SGD: every epoch,
shuffle with `rng.permutation`, then take one step per mini-batch. After each epoch, append the **full**
training loss to a list. Return `(params, losses)`.

Also write `accuracy(params, X, y)`: the fraction of rows whose largest logit is the true class.

In [ ]:
def accuracy(params, X, y):
    # TODO
    pass

def train_numpy(sizes, epochs=30, lr=0.1, batch=32, seed=0):
    # TODO
    pass

In [ ]:
# Check A3 (takes a few seconds)
t0 = time.time()
params_np, losses_np = train_numpy([64, 64, 10])
train_acc, test_acc = accuracy(params_np, X_train, y_train), accuracy(params_np, X_test, y_test)
print(f"train acc {train_acc:.3f}, test acc {test_acc:.3f}, {time.time() - t0:.1f} s")

plt.plot(range(1, len(losses_np) + 1), losses_np, marker="o")
plt.xlabel("epoch"); plt.ylabel("training loss"); plt.title("NumPy MLP on 8x8 digits"); plt.show()

assert len(losses_np) == 30 and losses_np[-1] < losses_np[0], "record one decreasing loss per epoch"
assert test_acc >= 0.95, f"expected at least 95% test accuracy, got {test_acc:.3f}"
print("OK")

## Part B: PyTorch

Colab has PyTorch and torchvision preinstalled. Locally, install them first (for example `uv add torch
torchvision`, or `pip install torch torchvision` in your environment). The next cell prints the version
and the device: `cuda` on a Colab GPU runtime, otherwise `cpu`. Both work.

In [ ]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print(torch.__version__, device)

### Task B1: the same digits network in PyTorch

Re-implement Part A's network with PyTorch:

1. A class `MLP(nn.Module)` whose constructor takes a list of sizes such as `[64, 64, 10]` and builds
   `nn.Linear` layers with `nn.ReLU()` between them (none after the last). Build the list of layers in a
   loop and wrap it in `nn.Sequential(*layers)`, so any number of layers works.
2. `train_torch(sizes, epochs=30, lr=0.1, batch=32, seed=0)`: call `torch.manual_seed(seed)` first, build
   the model, a `DataLoader` over `X_train, y_train` (as `float32` and `int64` tensors) with
   `shuffle=True`, `nn.CrossEntropyLoss` and `torch.optim.SGD`, and run the four-line training loop.
   Return the model.
3. `torch_accuracy(model, X, y)`: accuracy on NumPy arrays `X, y`, computed under `model.eval()` and
   `torch.no_grad()`.

This part is small enough for the CPU, so you can ignore `device` here.

In [ ]:
class MLP(nn.Module):
    def __init__(self, sizes):
        super().__init__()
        # TODO

    def forward(self, x):
        # TODO
        pass

def train_torch(sizes, epochs=30, lr=0.1, batch=32, seed=0):
    # TODO
    pass

def torch_accuracy(model, X, y):
    # TODO
    pass

In [ ]:
# Check B1
_m = MLP([64, 32, 16, 10])
assert sum(isinstance(l, nn.Linear) for l in _m.modules()) == 3 and sum(isinstance(l, nn.ReLU) for l in _m.modules()) == 2, "three Linear layers with ReLU between them, none at the end"
assert _m(torch.zeros(4, 64)).shape == (4, 10)
t0 = time.time()
model_digits = train_torch([64, 64, 10])
acc_t = torch_accuracy(model_digits, X_test, y_test)
print(f"PyTorch test acc {acc_t:.3f} (NumPy: {test_acc:.3f}), {time.time() - t0:.1f} s")
assert acc_t >= 0.95, f"expected at least 95% test accuracy, got {acc_t:.3f}"
print("OK")

### Task B2: MNIST to 97%

MNIST has 60,000 training and 10,000 test images of 28×28 digits. The given cell downloads it with
torchvision (about 12 MB of compressed downloads) and normalises the pixels with MNIST's mean and
standard deviation.

In [ ]:
from torchvision import datasets, transforms

tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
mnist_train = datasets.MNIST("data", train=True, download=True, transform=tfm)
mnist_test = datasets.MNIST("data", train=False, download=True, transform=tfm)
image, label = mnist_train[0]
print(len(mnist_train), len(mnist_test), image.shape, label)

Now train an MLP on it:

1. Call `torch.manual_seed(0)`. Make a training `DataLoader` with `batch_size=128`, `shuffle=True` and
   `generator=torch.Generator().manual_seed(0)` (this fixes the shuffling order), and a test `DataLoader`
   with `batch_size=1000`.
2. Build `model = nn.Sequential(nn.Flatten(), MLP([784, 256, 128, 10])).to(device)`. `nn.Flatten()` turns
   each `(1, 28, 28)` image into a vector of 784.
3. Train for 5 epochs with `torch.optim.Adam(model.parameters(), lr=1e-3)` and `nn.CrossEntropyLoss()`.
   Move every batch to the device with `xb, yb = xb.to(device), yb.to(device)`.
4. Write `evaluate(model, loader)` returning the test accuracy (use `model.eval()` and `torch.no_grad()`),
   and print the training loss and the test accuracy after every epoch.

Store the final test accuracy in `mnist_acc`. With these settings it reaches about 97.5 to 98% (results
vary slightly between CPU and GPU).

In [ ]:
def evaluate(model, loader):
    # TODO
    pass

# TODO: loaders, model, optimiser, 5 epochs of training
mnist_acc = None

In [ ]:
# Check B2
assert mnist_acc is not None, "store the final test accuracy in mnist_acc"
print(f"MNIST test accuracy: {mnist_acc:.4f}")
assert mnist_acc >= 0.97, "expected at least 97% test accuracy"
print("OK")

### Task B3: look and explain

Answer in a markdown cell below, two or three sentences each:

1. Build a fresh, untrained model with the same seed and compute its loss on the **first training batch,
   before any update**. Compare it with $\ln 10 \approx 2.30$, the loss of a model that guesses
   uniformly. What does the comparison tell you?
2. Compute the **training** accuracy of your MNIST model (reuse `evaluate` with a loader over
   `mnist_train`). How large is the gap to the test accuracy, and is this model overfitting badly?
3. Show a few test images the model gets wrong (`plt.imshow(image[0], cmap="gray")`), with the true and
   predicted labels. Would you have got them right?

### Stretch

- Remove the `nn.ReLU()` layers and retrain. What test accuracy does a purely linear model reach, and why?
- Add `nn.Dropout(0.2)` after each ReLU and `weight_decay=1e-4` to Adam. Does the gap from question 2
  shrink?
- Write a tiny NumPy version of `nn.Dropout` into your Part A network and train with it.

_Your answers to Task B3 here._